In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# EDA (Exploratory Data Analysis)

In [ ]:
from DB_pipeline.drive_auth_setup import get_drive_csv_as_df
df_hotel_reviews = get_drive_csv_as_df()

df_hotel_reviews.head()

In [ ]:
# columns
print(f"Total Number of columns: {len(df_hotel_reviews.columns.to_list())}")
df_hotel_reviews.columns.to_list()

In [ ]:
df_hotel_reviews.info()

### Describe() on continuous columns

In [ ]:
df_hotel_reviews.describe().T

### Describe() on object columns

In [ ]:
df_hotel_reviews.describe(include='object').T

## Analysis of column "REVIEW_SCORE"

In [ ]:
df_hotel_reviews[['Reviewer_Score']].describe().T

In [ ]:
# Plotly Express (px) is a high-level API in the Plotly library.
import plotly.express as px
# Review score Distribution
fig = px.histogram(df_hotel_reviews,x="Reviewer_Score",title="Review Score Distribution", nbins=20, text_auto=True)
fig.show()     # text_auto is used to adds numeric labels on top of each histogram bar.

# Data Preprocessing

## Cleaning Review Text
#### There are some values like "No Positive" in positive Review and
#### "No Negative" in negative review which can be counted as opposite sentiment, hence replacing those phrases

In [ ]:
df_hotel_reviews[['Negative_Review']].head()

####  here in above Negative Review column record at index 1 is No Negative

In [ ]:
# strip() trailing whitespace from review
df_hotel_reviews['Negative_Review'] = df_hotel_reviews['Negative_Review'].str.strip()

In [ ]:
df_hotel_reviews["Positive_Review"][0]

In [ ]:
# strip() trailing whitespace from review
df_hotel_reviews['Positive_Review'] = df_hotel_reviews['Positive_Review'].str.strip()
df_hotel_reviews['Positive_Review'][0]

## Replace 'No Negative' and 'No Positive' placeholder with ''

In [ ]:
df_hotel_reviews["Negative_Review"][1]

In [ ]:
df_hotel_reviews.loc[:, 'Positive_Review'] = df_hotel_reviews.Positive_Review.apply(lambda x: x.replace('No Positive', ''))
df_hotel_reviews.loc[:, 'Negative_Review'] = df_hotel_reviews.Negative_Review.apply(lambda x: x.replace('No Negative', ''))

In [ ]:
df_hotel_reviews['Negative_Review'][1]

## Merge Feature (Both Review Text)

#### Here, I merge text from both Positive and Negative Review columns to merge into a single text

In [ ]:
df_hotel_reviews[['Positive_Review']].head(1)

In [ ]:
df_hotel_reviews[['Negative_Review']].head(1)

In [ ]:
df_hotel_reviews['review'] = df_hotel_reviews['Positive_Review'] +" "+df_hotel_reviews['Negative_Review']

In [ ]:
pd.set_option('display.max_colwidth', None)  # it is used to set df column width to max (align to content of columm)
df_hotel_reviews[['review']].head(2)

## Set Sentiment Type
#### setting the sentiment threshold to 7, everything is negative below 7 and rest is Positive


In [ ]:
df_hotel_reviews["review_type"] = df_hotel_reviews["Reviewer_Score"] \
                                .apply(lambda x: "bad" if x < 7 else "good")

In [ ]:
df_reviews = df_hotel_reviews[["review", "review_type"]]

## Final Preprocessed Dataset

In [ ]:
df_reviews.head()

## Review Type Distribution Analysis

In [ ]:
fig = px.histogram(df_reviews, x="review_type", title='Review Type Distribution', text_auto=True)
fig.show()


#### Texts labeled as **`Good`**

In [ ]:

df_reviews[df_reviews.review_type == 'good'].review.value_counts()

#### Texts labeled as **`Bad`**

In [ ]:
df_reviews[df_reviews.review_type == 'bad'].review.value_counts()

In [ ]:
good_reviews = df_reviews[df_reviews.review_type == "good"]
bad_reviews = df_reviews[df_reviews.review_type == "bad"]
print(f"Good Reviews count\n: {len(good_reviews)}")
print(f"Bad Reviews count\n: {len(bad_reviews)}")

## Resample Dataset

Under sample the positive review to achieve a balanced distribution between reviews

In [ ]:
good_df = good_reviews.sample(n=len(bad_reviews), random_state=42)

df_review_resampled = pd.concat([good_df, bad_reviews]).reset_index(drop=True)
df_review_resampled.shape

In [ ]:
df_review_resampled.head()

In [ ]:
df_review_resampled.tail()

## After Resampled
### Review Type Distribution Analysis

In [ ]:

fig = px.histogram(df_review_resampled, x="review_type", title='Review Type Distribution (remsampled)', text_auto=True)
fig.show()


# Data Preparation

### Encoding Labels

In [ ]:
from sklearn.preprocessing import LabelEncoder
label_enc = LabelEncoder()
encoded_review = label_enc.fit_transform(df_review_resampled.review_type.values)

In [ ]:
# after label encoding the unique value of encoded_review
print([int(x) for x in set(encoded_review)])

# Train/test Split

In [ ]:
from sklearn.model_selection import train_test_split
train_reviews, test_reviews, y_train, y_test = train_test_split(
    df_review_resampled.review,
    encoded_review,
    test_size=0.25,
    random_state=42
  )

# Feature Engineering

## 1) TF-IDF

Term Frequency (TF):  
Measures how often a word appears in a document.
Example: In “The cat sat on the mat”, the word “cat” appears once out of six words → TF = 
1
/
6
.

Inverse Document Frequency (IDF):  
Reduces the weight of common words (like “the”) and increases the weight of rare words.
Formula:


![image.png](attachment:image.png)

| **Parameter** | **Meaning** | **Significance in Classification** |
| --- | --- | --- |
| ``min_df`` | Minimum number of documents a term must appear in | Filters out rare words that may add noise |
| ``max_df`` | Maximum proportion of documents a term can appear in | Removes overly common words (like stopwords) |
| ``ngram_range`` | Range of n‑grams (e.g., (1,2) → unigrams + bigrams) | Captures word sequences, improving context |
| ``stop_words`` | List of words to ignore | Prevents uninformative words from skewing results |
| ``max_features`` | Maximum number of features to keep | Controls dimensionality, avoids overfitting |
| ``norm`` | Normalization method (``l1``, ``l2``) | Ensures feature vectors are comparable in magnitude |
| ``use_idf`` | Whether to apply IDF weighting | If False, only TF is used (less discriminative) |
| ``smooth_idf`` | Adds 1 to document frequency before computing IDF | Prevents division by zero, stabilizes scores |
| ``sublinear_tf`` | Applies logarithmic scaling to TF | Reduces bias from very frequent words |

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfv = TfidfVectorizer(min_df=3,  max_features=None,
            strip_accents='unicode', analyzer='word',token_pattern=r'\w{1,}',
            ngram_range=(1, 3), use_idf=True,smooth_idf=True,sublinear_tf=True,
            stop_words = 'english')

tfv.fit(list(train_reviews) + list(test_reviews))

X_train_tfv =  tfv.transform(train_reviews)
X_test_tfv = tfv.transform(test_reviews)

# 2) Count Vectorizer
Count Vectorizer converts text into numerical vectors by counting how many times each word (or n‑gram) appears in a document.

| **Parameter** | **Meaning** | **Significance in Feature Engineering** |
| --- | --- | --- |
| ``min_df`` | Minimum number of documents a term must appear in | Filters out extremely rare words that add noise and don’t generalize well |
| ``max_df`` | Maximum proportion of documents a term can appear in | Removes overly common words (like “the”, “is”) that don’t help classification |
| ``ngram_range`` | Range of n‑grams to include (e.g., (1,2) → unigrams + bigrams) | Captures word sequences, improving context (e.g., “not good” vs “good”) |
| ``max_features`` | Maximum number of features (vocabulary size) | Controls dimensionality, prevents overfitting, keeps models efficient |
| ``stop_words`` | List of words to ignore | Prevents uninformative words from skewing results |
| ``binary`` | If True, records presence/absence instead of counts | Useful when frequency doesn’t matter, only occurrence |
| ``lowercase`` | Converts all text to lowercase | Ensures consistency, avoids treating “Cat” and “cat” as different |
| ``token_pattern`` | Regex for tokenization | Defines what counts as a “word” (e.g., alphanumeric only) |
| ``analyzer`` | Function to split text (``word``, ``char``, ``char_wb``) | Controls granularity: word‑level vs character‑level features |
| ``dtype`` | Data type of output matrix (default float64) | Impacts memory usage and computational efficiency |
| ``vocabulary`` | Predefined mapping of terms to indices | Fixes vocabulary across datasets, useful for production pipelines |

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
ctv = CountVectorizer(analyzer='word',token_pattern=r'\w{1,}',
            ngram_range=(1, 3), stop_words = 'english')

# Fitting Count Vectorizer to both training and test sets (semi-supervised learning)
ctv.fit(list(train_reviews) + list(test_reviews))

X_train_ctv=  ctv.transform(train_reviews)
X_test_ctv = ctv.transform(test_reviews)

### utility Function 
This function can be used to apply each classifcation model by passing only parameters to this function 

In [ ]:
from sklearn.metrics import classification_report, log_loss,roc_auc_score
def model_predict(clf, X_train,y_train, X_test, y_test):
    clf.fit(X_train, y_train)
    predictions = clf.predict_proba(X_test)

    print (f"logloss\n: {log_loss(y_test, predictions):0.3f}")
    print (f"ROC AUC\n: {roc_auc_score(y_test, predictions[:, 1]):0.3f}")
    print("classifcation Report:\n", classification_report(y_test, predictions[:, 1] > 0.5))

# Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(C=1.0, solver="saga", max_iter=1000)

print("Logistic Regression using countvectorizer \n")
model_predict(clf, X_train_ctv,y_train, X_test_ctv, y_test)

print("Logistic Regression using tf_idf \n")
model_predict(clf,X_train_tfv,y_train,X_test_tfv,y_test)



# Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB
clf = MultinomialNB()

print("Naive Bayes using countvectorizer \n")
model_predict(clf, X_train_ctv,y_train, X_test_ctv, y_test)

print("Naive Bayes using tf_idf \n")
model_predict(clf,X_train_tfv,y_train,X_test_tfv,y_test)


# Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
clf = DecisionTreeClassifier(max_depth=20, random_state=42)

print("Decision Tree using countvectorizer \n")
model_predict(clf, X_train_ctv,y_train, X_test_ctv, y_test)

print("Decision Tree using tf_idf \n")
model_predict(clf,X_train_tfv,y_train,X_test_tfv,y_test)


# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

print("Random Forest using countvectorizer \n")
model_predict(clf, X_train_ctv,y_train, X_test_ctv, y_test)

print("Random Forest using tf_idf \n")
model_predict(clf,X_train_tfv,y_train,X_test_tfv,y_test)


# KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
clf = KNeighborsClassifier(n_neighbors=15, n_jobs=-1)

print("KNN using countvectorizer \n")
model_predict(clf, X_train_ctv,y_train, X_test_ctv, y_test)

print("KNN using tf_idf \n")
model_predict(clf,X_train_tfv,y_train,X_test_tfv,y_test)


# Support Vector Machine (SVM)
#### `LinearSVC` doesn't support `predict_proba` directly (needed by our `model_predict` utility function above), so it's wrapped in `CalibratedClassifierCV` to get probability estimates without changing the function itself.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
clf = CalibratedClassifierCV(LinearSVC(max_iter=2000))

print("SVM using countvectorizer \n")
model_predict(clf, X_train_ctv,y_train, X_test_ctv, y_test)

print("SVM using tf_idf \n")
model_predict(clf,X_train_tfv,y_train,X_test_tfv,y_test)


# XGBoost

In [ ]:
import xgboost as xgb
clf = xgb.XGBClassifier(n_estimators=200, eval_metric='logloss', n_jobs=-1, random_state=42)

print("XGBoost using countvectorizer \n")
model_predict(clf, X_train_ctv,y_train, X_test_ctv, y_test)

print("XGBoost using tf_idf \n")
model_predict(clf,X_train_tfv,y_train,X_test_tfv,y_test)


# LightGBM

In [ ]:
import lightgbm as lgb
clf = lgb.LGBMClassifier(n_estimators=200, n_jobs=-1, random_state=42)

print("LightGBM using countvectorizer \n")
model_predict(clf, X_train_ctv,y_train, X_test_ctv, y_test)

print("LightGBM using tf_idf \n")
model_predict(clf,X_train_tfv,y_train,X_test_tfv,y_test)


# Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
clf = GradientBoostingClassifier(n_estimators=100, random_state=42)

print("Gradient Boosting using countvectorizer \n")
model_predict(clf, X_train_ctv,y_train, X_test_ctv, y_test)

print("Gradient Boosting using tf_idf \n")
model_predict(clf,X_train_tfv,y_train,X_test_tfv,y_test)


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=45db2752-7c95-4900-a37b-d4cfad3dfece' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>